In [37]:
library(readxl)   
library(dplyr)    
library(tidyr)    
library(ggplot2)  

In [38]:
bls_raw <- read_excel(
  "~/Desktop/Project-Econ-326-repo/Econ326_Data/national_M2025_dl.xlsx")

ai_raw <- read_excel(
  "~/Desktop/Project-Econ-326-repo/Econ326_Data/AIOE_DataAppendix (2).xlsx",
  sheet = "Appendix A")

skills_raw <- read_excel(
  "~/Desktop/Project-Econ-326-repo/Econ326_Data/db_27_0_excel/Skills.xlsx")

crosswalk_raw <- read_excel(
  "~/Desktop/Project-Econ-326-repo/Econ326_Data/2010_to_2019_Crosswalk (2).xlsx",
  skip = 3)

In [40]:
bls_selected <- bls_raw %>%
  filter(
    AREA == "99",
    NAICS == "000000",
    OWN_CODE == "1235",
    O_GROUP == "detailed") %>%
  select(soc = OCC_CODE,
    occupation = OCC_TITLE,
    wage = A_MEAN,
    employment = TOT_EMP)

nrow(bls_selected)

[1] 830

In [41]:
bls_selected %>%
  filter(
    is.na(suppressWarnings(as.numeric(wage)))
  ) %>%
  select(soc, occupation, wage)

soc,occupation,wage
<chr>,<chr>,<chr>
27-2011,Actors,*
27-2031,Dancers,*
27-2042,Musicians and Singers,*
27-2091,"Disc Jockeys, Except Radio",*
27-2099,"Entertainers and Performers, Sports and Related Workers, All Other",*


In [47]:
bls <- bls_selected %>%
  mutate(
    wage = as.numeric(wage),
    employment = as.numeric(employment)
  )
head(bls)

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `wage = as.numeric(wage)`.
Caused by warning:
! NAs introduced by coercion”


soc,occupation,wage,employment
<chr>,<chr>,<dbl>,<dbl>
11-1011,Chief Executives,269630,204350
11-1021,General and Operations Managers,134940,3503020
11-2011,Advertising and Promotions Managers,154280,21470
11-2021,Marketing Managers,177770,395240
11-2022,Sales Managers,164350,637080
11-2032,Public Relations Managers,164760,74850


In [48]:
ai <- ai_raw %>%
  select(
    soc_2010 = `SOC Code`,
    AI_exposure = AIOE )

head(ai)

soc_2010,AI_exposure
<chr>,<dbl>
11-1011,1.3342460
11-1021,0.5748773
11-2011,1.2943870
11-2021,1.3150320
11-2022,1.2662800
11-2031,1.2936890


In [111]:
crosswalk <- crosswalk_raw %>%
  select(
    code_2010 = `O*NET-SOC 2010 Code`,
    code_2019 = `O*NET-SOC 2019 Code`)

head(crosswalk)

code_2010,code_2019
<chr>,<chr>
11-1011.00,11-1011.00
11-1011.03,11-1011.03
11-1021.00,11-1021.00
11-1031.00,11-1031.00
11-2011.00,11-2011.00
11-2011.01,11-2011.00


In [112]:
ai <- ai_raw %>%
  select(
    soc_2010 = `SOC Code`,
    AI_exposure = AIOE
  )

In [113]:
crosswalk_raw <- read_excel(
  "~/Desktop/Project-Econ-326-repo/Econ326_Data/2010_to_2019_Crosswalk (2).xlsx",
  skip = 3)

head(crosswalk_raw)

O*NET-SOC 2010 Code,O*NET-SOC 2010 Title,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title
<chr>,<chr>,<chr>,<chr>
11-1011.00,Chief Executives,11-1011.00,Chief Executives
11-1011.03,Chief Sustainability Officers,11-1011.03,Chief Sustainability Officers
11-1021.00,General and Operations Managers,11-1021.00,General and Operations Managers
11-1031.00,Legislators,11-1031.00,Legislators
11-2011.00,Advertising and Promotions Managers,11-2011.00,Advertising and Promotions Managers
11-2011.01,Green Marketers,11-2011.00,Advertising and Promotions Managers


In [114]:
ccrosswalk <- crosswalk_raw %>%
  select(
    code_2010 = `O*NET-SOC 2010 Code`,
    code_2019 = `O*NET-SOC 2019 Code`
  )


crosswalk <- crosswalk %>%
  mutate(
    code_2010 = substr(code_2010, 1, 7),
    code_2019 = substr(code_2019, 1, 7))

crosswalk <- distinct(crosswalk)

head(crosswalk)

code_2010,code_2019
<chr>,<chr>
11-1011,11-1011
11-1021,11-1021
11-1031,11-1031
11-2011,11-2011
11-2021,11-2021
11-2022,11-2022


In [115]:
crosswalk <- distinct(crosswalk)

crosswalk <- crosswalk %>%
  group_by(code_2010) %>%
  mutate(new_matches = n()) %>%
  ungroup()

crosswalk <- crosswalk %>%
  group_by(code_2019) %>%
  mutate(old_matches = n()) %>%
  ungroup()

In [119]:
stable_codes <- crosswalk %>%
  filter(
    new_matches == 1,
    old_matches == 1,
    code_2010 == code_2019
  )

bls_stable <- merge(
  bls,
  stable_codes,
  by.x = "soc",
  by.y = "code_2019"
)


In [120]:
stable_codes <- crosswalk %>%
  filter(
    new_matches == 1,
    old_matches == 1,
    code_2010 == code_2019
  )

bls_stable <- merge(
  bls,
  stable_codes,
  by.x = "soc",
  by.y = "code_2019"
)

matched_ai <- merge(
  bls_stable,
  ai,
  by.x = "code_2010",
  by.y = "soc_2010"
)

matched_ai_usable <- matched_ai %>%
  filter(
    !is.na(wage), wage > 0,
    !is.na(employment), employment > 0,
    !is.na(AI_exposure)
  )

In [122]:
skills <- skills_raw %>%
  filter(`Scale ID` == "IM")

skills <- skills %>%
  filter(`Element Name` %in% c(
    "Speaking",
    "Active Listening",
    "Critical Thinking",
    "Complex Problem Solving",
    "Judgment and Decision Making"
  ))

skills <- skills %>%
  select(
    soc = `O*NET-SOC Code`,
    skill = `Element Name`,
    rating = `Data Value`
  )

skills$soc <- substr(skills$soc, 1, 7)

skills_average <- skills %>%
  group_by(soc, skill) %>%
  summarise(
    rating = mean(rating),
    .groups = "drop"
  )

head(skills_average)

soc,skill,rating
<chr>,<chr>,<dbl>
11-1011,Active Listening,4.06
11-1011,Complex Problem Solving,4.19
11-1011,Critical Thinking,4.25
11-1011,Judgment and Decision Making,4.19
11-1011,Speaking,4.19
11-1021,Active Listening,4.00


In [123]:
library(tidyr)

skills_wide <- skills_average %>%
  pivot_wider(
    names_from = skill,
    values_from = rating
  )

head(skills_wide)

soc,Active Listening,Complex Problem Solving,Critical Thinking,Judgment and Decision Making,Speaking
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
11-1011,4.06,4.19,4.25,4.19,4.19
11-1021,4.00,3.50,3.88,3.50,4.00
11-2011,4.12,3.50,4.00,3.75,4.00
11-2021,3.88,3.62,3.88,3.75,3.88
11-2022,4.00,3.75,3.88,3.75,4.00
11-3012,4.00,3.25,3.75,3.25,3.88


In [139]:
matched_skills <- merge(
  matched_ai_usable,
  skills_wide,
  by = "soc",
  all.x = TRUE
)

analysis_data <- matched_skills %>%
  drop_na(
    Speaking,
    `Active Listening`,
    `Critical Thinking`,
    `Complex Problem Solving`,
    `Judgment and Decision Making`
  )

analysis_data %>%
  summarise(
    N = n(),
    Mean = mean(wage),
    SD = sd(wage),
    Min = min(wage),
    Max = max(wage)
  )

N,Mean,SD,Min,Max
<int>,<dbl>,<dbl>,<dbl>,<dbl>
624,74421.17,37793.98,31980,346490


In [140]:
analysis_data <- analysis_data %>%
  mutate(
    log_wage = log(wage),
    log_employment = log(employment)
  )

analysis_data %>%
  summarise(
    Mean = mean(log_wage),
    SD = sd(log_wage),
    Min = min(log_wage),
    Max = max(log_wage)
  )

Mean,SD,Min,Max
<dbl>,<dbl>,<dbl>,<dbl>
11.12097,0.4192232,10.37287,12.75561


In [141]:
analysis_data <- analysis_data %>%
  mutate(AI = (AI_exposure - mean(AI_exposure)) / sd(AI_exposure) )

analysis_data %>%
  summarise(
    Mean = mean(AI),
    SD = sd(AI),
    Min = min(AI),
    Max = max(AI)
  )

Mean,SD,Min,Max
<dbl>,<dbl>,<dbl>,<dbl>
-3.437421e-16,1,-2.032543,1.573079


In [142]:
analysis_data <- analysis_data %>%
  mutate(
    Speaking_z =
      (Speaking - mean(Speaking)) / sd(Speaking),

    Listening_z =
      (`Active Listening` - mean(`Active Listening`)) /
      sd(`Active Listening`),

    CriticalThinking_z =
      (`Critical Thinking` - mean(`Critical Thinking`)) /
      sd(`Critical Thinking`),

    ProblemSolving_z =
      (`Complex Problem Solving` - mean(`Complex Problem Solving`)) /
      sd(`Complex Problem Solving`),

    Judgment_z =
      (`Judgment and Decision Making` -
         mean(`Judgment and Decision Making`)) /
      sd(`Judgment and Decision Making`)
  )

analysis_data <- analysis_data %>%
  mutate(
    skills_average = (
      Speaking_z +
      Listening_z +
      CriticalThinking_z +
      ProblemSolving_z +
      Judgment_z
    ) / 5 )

In [152]:
analysis_data <- analysis_data %>%
  mutate(Skills = (skills_average - mean(skills_average)) /
      sd(skills_average) )

analysis_data %>%
  summarise(
    Mean = mean(Skills),
    SD = sd(Skills),
    Min = min(Skills),
    Max = max(Skills)
  )

Mean,SD,Min,Max
<dbl>,<dbl>,<dbl>,<dbl>
-1.069301e-16,1,-2.714252,2.661249


In [144]:
analysis_data <- analysis_data %>%
  mutate(
    communication_average =
      (Speaking_z + Listening_z) / 2,

    problem_average =
      (CriticalThinking_z + ProblemSolving_z + Judgment_z) / 3
  )

analysis_data <- analysis_data %>%
  mutate(
    Communication =
      (communication_average - mean(communication_average)) /
      sd(communication_average),

    ProblemSolving =
      (problem_average - mean(problem_average)) /
      sd(problem_average)
  )

analysis_data %>%
  summarise(
    Communication_mean = mean(Communication),
    Communication_SD = sd(Communication),
    ProblemSolving_mean = mean(ProblemSolving),
    ProblemSolving_SD = sd(ProblemSolving)
  )

Communication_mean,Communication_SD,ProblemSolving_mean,ProblemSolving_SD
<dbl>,<dbl>,<dbl>,<dbl>
1.492396e-15,1,-3.227475e-16,1


In [145]:
analysis_data %>%
  summarise(
    Mean = mean(ProblemSolving),
    SD = sd(ProblemSolving)
  )

Mean,SD
<dbl>,<dbl>
-3.227475e-16,1


In [146]:
analysis_data <- analysis_data %>%
  separate( soc,into = c("occupation_group", "occupation_detail"),
    sep = "-", remove = FALSE) 

In [147]:
analysis_data <- analysis_data %>%
  mutate(
    AI_squared = AI^2,
    Skills_squared = Skills^2,

    Management = as.numeric(occupation_group == "11"),
    Business = as.numeric(occupation_group == "13"),
    Computer = as.numeric(occupation_group == "15"),
    Engineering = as.numeric(occupation_group == "17"),
    Science = as.numeric(occupation_group == "19"),
    Community = as.numeric(occupation_group == "21"),
    Legal = as.numeric(occupation_group == "23"),
    Education = as.numeric(occupation_group == "25"),
    Arts = as.numeric(occupation_group == "27"),
    Healthcare = as.numeric(occupation_group == "29"),
    Healthcare_support = as.numeric(occupation_group == "31"),
    Protective_service = as.numeric(occupation_group == "33"),
    Food_service = as.numeric(occupation_group == "35"),
    Cleaning = as.numeric(occupation_group == "37"),
    Personal_care = as.numeric(occupation_group == "39"),
    Sales = as.numeric(occupation_group == "41"),
    Office = as.numeric(occupation_group == "43"),
    Farming = as.numeric(occupation_group == "45"),
    Construction = as.numeric(occupation_group == "47"),
    Maintenance = as.numeric(occupation_group == "49"),
    Production = as.numeric(occupation_group == "51"),
    Transportation = as.numeric(occupation_group == "53")
  )

In [148]:
wage_summary <- analysis_data %>%
  summarise(
    Variable = "Log annual wage",
    N = n(),
    Mean = mean(log_wage),
    SD = sd(log_wage),
    Min = min(log_wage),
    Max = max(log_wage)
  )

ai_summary <- analysis_data %>%
  summarise(
    Variable = "AI exposure",
    N = n(),
    Mean = mean(AI),
    SD = sd(AI),
    Min = min(AI),
    Max = max(AI)
  )

skills_summary <- analysis_data %>%
  summarise(
    Variable = "Human skills",
    N = n(),
    Mean = mean(Skills),
    SD = sd(Skills),
    Min = min(Skills),
    Max = max(Skills)
  )

employment_summary <- analysis_data %>%
  summarise(
    Variable = "Log employment",
    N = n(),
    Mean = mean(log_employment),
    SD = sd(log_employment),
    Min = min(log_employment),
    Max = max(log_employment)
  )

ai_squared_summary <- analysis_data %>%
  summarise(
    Variable = "AI exposure squared",
    N = n(),
    Mean = mean(AI_squared),
    SD = sd(AI_squared),
    Min = min(AI_squared),
    Max = max(AI_squared)
  )

skills_squared_summary <- analysis_data %>%
  summarise(
    Variable = "Human skills squared",
    N = n(),
    Mean = mean(Skills_squared),
    SD = sd(Skills_squared),
    Min = min(Skills_squared),
    Max = max(Skills_squared)
  )

communication_summary <- analysis_data %>%
  summarise(
    Variable = "Communication",
    N = n(),
    Mean = mean(Communication),
    SD = sd(Communication),
    Min = min(Communication),
    Max = max(Communication)
  )

problem_solving_summary <- analysis_data %>%
  summarise(
    Variable = "Problem-solving",
    N = n(),
    Mean = mean(ProblemSolving),
    SD = sd(ProblemSolving),
    Min = min(ProblemSolving),
    Max = max(ProblemSolving)
  )

In [149]:
management_summary <- analysis_data %>%
  summarise(
    Variable = "Management",
    N = n(),
    Mean = mean(Management),
    SD = sd(Management),
    Min = min(Management),
    Max = max(Management)
  )

business_summary <- analysis_data %>%
  summarise(
    Variable = "Business and financial operations",
    N = n(),
    Mean = mean(Business),
    SD = sd(Business),
    Min = min(Business),
    Max = max(Business)
  )

computer_summary <- analysis_data %>%
  summarise(
    Variable = "Computer and mathematical",
    N = n(),
    Mean = mean(Computer),
    SD = sd(Computer),
    Min = min(Computer),
    Max = max(Computer)
  )

engineering_summary <- analysis_data %>%
  summarise(
    Variable = "Architecture and engineering",
    N = n(),
    Mean = mean(Engineering),
    SD = sd(Engineering),
    Min = min(Engineering),
    Max = max(Engineering)
  )

science_summary <- analysis_data %>%
  summarise(
    Variable = "Life, physical, and social science",
    N = n(),
    Mean = mean(Science),
    SD = sd(Science),
    Min = min(Science),
    Max = max(Science)
  )

community_summary <- analysis_data %>%
  summarise(
    Variable = "Community and social service",
    N = n(),
    Mean = mean(Community),
    SD = sd(Community),
    Min = min(Community),
    Max = max(Community)
  )

legal_summary <- analysis_data %>%
  summarise(
    Variable = "Legal",
    N = n(),
    Mean = mean(Legal),
    SD = sd(Legal),
    Min = min(Legal),
    Max = max(Legal)
  )

education_summary <- analysis_data %>%
  summarise(
    Variable = "Educational instruction and library",
    N = n(),
    Mean = mean(Education),
    SD = sd(Education),
    Min = min(Education),
    Max = max(Education)
  )

arts_summary <- analysis_data %>%
  summarise(
    Variable = "Arts, design, entertainment, sports, and media",
    N = n(),
    Mean = mean(Arts),
    SD = sd(Arts),
    Min = min(Arts),
    Max = max(Arts)
  )

healthcare_summary <- analysis_data %>%
  summarise(
    Variable = "Healthcare practitioners and technical",
    N = n(),
    Mean = mean(Healthcare),
    SD = sd(Healthcare),
    Min = min(Healthcare),
    Max = max(Healthcare)
  )

healthcare_support_summary <- analysis_data %>%
  summarise(
    Variable = "Healthcare support",
    N = n(),
    Mean = mean(Healthcare_support),
    SD = sd(Healthcare_support),
    Min = min(Healthcare_support),
    Max = max(Healthcare_support)
  )

protective_service_summary <- analysis_data %>%
  summarise(
    Variable = "Protective service",
    N = n(),
    Mean = mean(Protective_service),
    SD = sd(Protective_service),
    Min = min(Protective_service),
    Max = max(Protective_service)
  )

food_service_summary <- analysis_data %>%
  summarise(
    Variable = "Food preparation and serving",
    N = n(),
    Mean = mean(Food_service),
    SD = sd(Food_service),
    Min = min(Food_service),
    Max = max(Food_service)
  )

cleaning_summary <- analysis_data %>%
  summarise(
    Variable = "Building and grounds cleaning and maintenance",
    N = n(),
    Mean = mean(Cleaning),
    SD = sd(Cleaning),
    Min = min(Cleaning),
    Max = max(Cleaning)
  )

personal_care_summary <- analysis_data %>%
  summarise(
    Variable = "Personal care and service",
    N = n(),
    Mean = mean(Personal_care),
    SD = sd(Personal_care),
    Min = min(Personal_care),
    Max = max(Personal_care)
  )

sales_summary <- analysis_data %>%
  summarise(
    Variable = "Sales",
    N = n(),
    Mean = mean(Sales),
    SD = sd(Sales),
    Min = min(Sales),
    Max = max(Sales)
  )

office_summary <- analysis_data %>%
  summarise(
    Variable = "Office and administrative support",
    N = n(),
    Mean = mean(Office),
    SD = sd(Office),
    Min = min(Office),
    Max = max(Office)
  )

farming_summary <- analysis_data %>%
  summarise(
    Variable = "Farming, fishing, and forestry",
    N = n(),
    Mean = mean(Farming),
    SD = sd(Farming),
    Min = min(Farming),
    Max = max(Farming)
  )

construction_summary <- analysis_data %>%
  summarise(
    Variable = "Construction and extraction",
    N = n(),
    Mean = mean(Construction),
    SD = sd(Construction),
    Min = min(Construction),
    Max = max(Construction)
  )

maintenance_summary <- analysis_data %>%
  summarise(
    Variable = "Installation, maintenance, and repair",
    N = n(),
    Mean = mean(Maintenance),
    SD = sd(Maintenance),
    Min = min(Maintenance),
    Max = max(Maintenance)
  )

production_summary <- analysis_data %>%
  summarise(
    Variable = "Production",
    N = n(),
    Mean = mean(Production),
    SD = sd(Production),
    Min = min(Production),
    Max = max(Production)
  )

transportation_summary <- analysis_data %>%
  summarise(
    Variable = "Transportation and material moving",
    N = n(),
    Mean = mean(Transportation),
    SD = sd(Transportation),
    Min = min(Transportation),
    Max = max(Transportation)
  )

In [150]:
summary_table <- bind_rows(
  wage_summary,
  ai_summary,
  skills_summary,
  employment_summary,
  ai_squared_summary,
  skills_squared_summary,
  communication_summary,
  problem_solving_summary,
  management_summary,
  business_summary,
  computer_summary,
  engineering_summary,
  science_summary,
  community_summary,
  legal_summary,
  education_summary,
  arts_summary,
  healthcare_summary,
  healthcare_support_summary,
  protective_service_summary,
  food_service_summary,
  cleaning_summary,
  personal_care_summary,
  sales_summary,
  office_summary,
  farming_summary,
  construction_summary,
 maintenance_summary,
  production_summary,
  transportation_summary
)

In [151]:
summary_display <- summary_table %>%
  mutate(
    Mean = round(Mean, 3),
    SD = round(SD, 3),
    Min = round(Min, 3),
    Max = round(Max, 3)
  )

print(summary_display, row.names = FALSE)

                                       Variable   N   Mean    SD    Min    Max
                                Log annual wage 624 11.121 0.419 10.373 12.756
                                    AI exposure 624  0.000 1.000 -2.033  1.573
                                   Human skills 624  0.000 1.000 -2.714  2.661
                                 Log employment 624 10.686 1.700  5.394 15.176
                            AI exposure squared 624  0.998 0.829  0.000  4.131
                           Human skills squared 624  0.998 1.016  0.000  7.367
                                  Communication 624  0.000 1.000 -3.053  2.485
                                Problem-solving 624  0.000 1.000 -2.693  2.710
                                     Management 624  0.042 0.200  0.000  1.000
              Business and financial operations 624  0.034 0.180  0.000  1.000
                      Computer and mathematical 624  0.005 0.069  0.000  1.000
                   Architecture and engineering 624 